<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L35_Gradient_Accumulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L35: Gradient Accumulation - Large Effective Batch Size

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 35 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Use gradient accumulation to simulate a larger batch size
- Configure Trainer with gradient_accumulation_steps
- Implement accumulation in a custom training loop

---

## Concept: Gradient Accumulation

**Gradient accumulation**: run N forward-backward passes with a small batch, accumulate gradients, then do one optimizer step. Effective batch size = per_device_batch_size * num_devices * gradient_accumulation_steps. Saves memory when you want a large effective batch.

---

In [ ]:
!pip install transformers torch datasets -q
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Trainer with gradient_accumulation_steps

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(400))
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
args = TrainingArguments(
    output_dir="./out_accum_l35",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()
print("Effective batch = 4 * 4 = 16 per device (no accumulation would be 4).")

## Part 2: Custom Loop with Accumulation

---

In [ ]:
from torch.utils.data import DataLoader

accum_steps = 4
model2 = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
optimizer = torch.optim.AdamW(model2.parameters(), lr=2e-5)
loader = DataLoader(train_ds, batch_size=4)
model2.train()
optimizer.zero_grad()
for i, batch in enumerate(loader):
    loss = model2(**batch).loss / accum_steps
    loss.backward()
    if (i + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
print("Custom accumulation: backward N times, then step once.")

## Exercises

1. Compare training loss curve: batch_size=16 vs batch_size=4 + accumulation_steps=4.
2. Increase accumulation until effective batch is 64; note memory and step count.
3. Combine gradient accumulation with fp16 and measure total memory.

---

## Key Takeaways

1. gradient_accumulation_steps multiplies effective batch size without increasing memory per step.
2. In custom loops: loss/accum_steps, backward, step every accum_steps.
3. Learning rate often tuned for effective batch size (e.g. linear scaling rule).

---

## Next Lesson

**L36: Advanced Optimization** – AdamW, schedules, warmup, clipping.

---